# Lesson 3 — The Randomness Dial

Greedy generation gets stuck in loops. Temperature reshapes the odds. This
notebook builds both and finds your machine's loop.

In [ ]:
# The corpus: Alice in Wonderland (public domain), with a built-in backup.
import urllib.request, re

FALLBACK = ("the small machine counted every letter of the paragraph and then began "
    "to write its own strange sentences about the city and the lake and the long "
    "quiet train ride home it wrote about the coach and the counselor and the "
    "quiet gym at seven in the morning and although every line was gibberish the "
    "shape of the words was english because the counts had come from english ") * 8

try:
    raw = urllib.request.urlopen("https://www.gutenberg.org/files/11/11-0.txt", timeout=15).read().decode("utf-8")
    raw = raw[raw.find("Alice was beginning"):raw.find("THE END")]
    print("Loaded Alice in Wonderland:", len(raw), "characters")
except Exception as e:
    raw = FALLBACK
    print("Download failed (%s) - using the built-in backup corpus." % type(e).__name__)

# Keep only lowercase letters and spaces - 27 symbols total.
corpus = re.sub(r"[^a-z ]+", " ", raw.lower())
corpus = re.sub(r" +", " ", corpus).strip()
print("Cleaned corpus:", len(corpus), "characters")
print(repr(corpus[:100]))

In [ ]:
counts = {}
for i in range(len(corpus) - 1):
    a, b = corpus[i], corpus[i + 1]
    counts.setdefault(a, {}).setdefault(b, 0)
    counts[a][b] += 1

probs = {a: {b: n / sum(row.values()) for b, n in row.items()}
         for a, row in counts.items()}

## Greedy: always pick the favorite

Deterministic — so the moment it revisits a letter, it repeats forever.
Let's catch the exact loop.

In [ ]:
def greedy(start="t", length=80):
    out = start
    cur = start
    for _ in range(length):
        row = probs.get(cur)
        if not row:
            break
        cur = max(row.items(), key=lambda kv: kv[1])[0]
        out += cur
    return out

line = greedy()
print(line)

# Find the cycle: walk favorites until a letter repeats.
seen, cur, path = {}, "t", "t"
while cur not in seen:
    seen[cur] = True
    cur = max(probs[cur].items(), key=lambda kv: kv[1])[0]
    path += cur
print("the loop it falls into:", path)

## Temperature

Reshape each probability with `p ** (1/T)`, then re-normalize.
Below 1: favorites get stronger. Above 1: odds flatten toward equal.

In [ ]:
import random

def generate(T=1.0, start="t", length=120):
    out = start
    cur = start
    for _ in range(length):
        row = probs.get(cur)
        if not row:
            cur = " "
            continue
        letters = list(row.keys())
        weights = [p ** (1.0 / T) for p in row.values()]
        cur = random.choices(letters, weights=weights)[0]
        out += cur
    return out

for T in [0.2, 1.0, 2.5]:
    print(f"--- temperature {T}")
    for _ in range(3):
        print(generate(T))

## Turn-in

Label each temperature's character in one honest sentence. Then: for a model
helping with math homework, which setting — and why?

In [ ]:
notes = """
T=0.2:
T=1.0:
T=2.5:
For math homework I would choose T=___ because:
"""
print(notes)